# 🔢 Semana 8 · Unidad 3 — Insertion Sort y Bubble Sort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 8 · Unidad 3 — Insertion Sort + Bubble Sort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

## Verificación de Dependencias

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {
    'numpy':      'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Implementar** Insertion Sort y explicar su invariante de ciclo paso a paso.
2. **Identificar** por qué Insertion Sort es O(n) en el mejor caso y O(n²) en el peor.
3. **Comparar** Insertion Sort y Bubble Sort en términos de comparaciones, swaps y estabilidad.
4. **Aplicar** la variante con *early-exit* de Bubble Sort para aprovechar datos casi ordenados.
5. **Elegir** el algoritmo elemental correcto según las características del problema.

# Sección 1: Motivación — El Problema de Ordenar Cartas (5 minutos)

## ¿Cómo ordenas tú una mano de cartas?

Imagina que recibes 7 cartas de a una. La mayoría de las personas hace esto naturalmente:

```
Cartas en mano:  [5♠]
Llega la carta:  [2♥] → insertar ANTES del 5 → [2♥, 5♠]
Llega la carta:  [8♣] → insertar DESPUÉS del 5 → [2♥, 5♠, 8♣]
Llega la carta:  [3♦] → insertar entre 2 y 5 → [2♥, 3♦, 5♠, 8♣]
```

Esto es exactamente **Insertion Sort**.

> 📌 **Definición:** Insertion Sort mantiene un prefijo ordenado `a[0..i-1]` e inserta el elemento  
> `a[i]` en su posición correcta dentro de ese prefijo, desplazando los mayores una posición a la derecha.

> 🎙️ **[PAUSA PROFESOR]** Pregunta: *"¿Alguien ordena sus cartas de otra forma? ¿Cómo funciona Selection Sort comparado con esto?"*

## La diferencia clave con Selection Sort

| Algoritmo | ¿Qué hace en cada pasada? | ¿Aprovecha orden previo? |
|-----------|--------------------------|--------------------------|
| Selection Sort | Busca el mínimo global en `a[i..n-1]` | ❌ No |
| Insertion Sort | Coloca `a[i]` en su lugar en `a[0..i]` | ✅ Sí |

# Sección 2: Insertion Sort (30 minutos)

In [ ]:
# Insertion Sort — implementación canónica con modo verbose
def insertion_sort(lista: list, verbose: bool = False) -> list:
    """
    Ordena 'lista' en orden no decreciente usando Insertion Sort.

    Idea: mantiene a[0..i-1] ordenado. En cada paso i, toma a[i]
    y lo 'inserta' en su posición correcta desplazando elementos mayores.

    Parámetros:
        lista  (list): lista de elementos comparables
        verbose (bool): si True, imprime el estado tras cada inserción

    Retorna:
        list: nueva lista ordenada (no modifica la original)

    Complejidad:
        Temporal: O(n²) peor caso (lista invertida)
                  O(n)  mejor caso (lista ya ordenada — solo comparaciones)
        Espacial: O(1) extra — in-place (sobre la copia)
    """
    a = lista[:]          # trabajamos sobre una copia
    n = len(a)
    comparaciones = 0
    desplazamientos = 0

    for i in range(1, n):
        clave = a[i]      # elemento a insertar
        j = i - 1         # índice del último elemento del prefijo ordenado

        # Desplazar a la derecha todos los elementos > clave
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]
            j -= 1
            comparaciones += 1
            desplazamientos += 1

        # Una comparación extra cuando el while terminó por a[j] <= clave
        if j >= 0:
            comparaciones += 1

        a[j + 1] = clave  # insertar en el hueco dejado

        if verbose:
            prefijo = a[:i+1]
            sufijo  = a[i+1:]
            print(f"  Paso {i}: clave={clave} → {prefijo} | {sufijo}  "
                  f"(cmp={comparaciones}, desp={desplazamientos})")

    return a, comparaciones, desplazamientos


# ─── Demo en clase ────────────────────────────────────────────────────────
print("=== Insertion Sort — trazado completo ===")
datos = [5, 2, 8, 3, 9, 1, 7]
print(f"Entrada: {datos}")
resultado, cmp, desp = insertion_sort(datos, verbose=True)
print(f"\nResultado: {resultado}")
print(f"Total: {cmp} comparaciones, {desp} desplazamientos")

In [ ]:
# Verificación de la invariante: a[0..i] está ordenado en cada paso
def verificar_invariante_insertion(lista):
    """Verifica que el prefijo a[0..i] esté ordenado tras cada inserción."""
    a = lista[:]
    n = len(a)
    for i in range(1, n):
        clave = a[i]
        j = i - 1
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = clave
        # Verificar invariante: a[0..i] ordenado
        prefijo = a[:i+1]
        assert prefijo == sorted(prefijo), f"¡Invariante rota en paso {i}!"
        print(f"  ✅ Paso {i}: {prefijo} está ordenado")
    return a

print("=== Verificación de invariante ===")
verificar_invariante_insertion([7, 3, 5, 1, 9, 2])
print("\n✔ Invariante de ciclo verificada en todos los pasos.")

## Visualización Animada — Insertion Sort

In [ ]:
# Animación de Insertion Sort con matplotlib
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import random

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if not EN_COLAB:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')

def animar_insertion_sort(datos_orig, interval=500):
    """Genera animación paso a paso de Insertion Sort."""
    a = datos_orig[:]
    n = len(a)
    frames = []

    # Capturar todos los estados
    frames.append((a[:], -1, -1, 0))   # estado inicial
    for i in range(1, n):
        clave = a[i]
        j = i - 1
        frames.append((a[:], i, -1, i))  # marca la clave
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]
            j -= 1
            frames.append((a[:], j + 1, i, i))
        a[j + 1] = clave
        frames.append((a[:], j + 1, -1, i))

    frames.append((a[:], -1, -1, n))   # estado final

    # Colores
    COLOR_BASE    = '#90CAF9'   # azul claro
    COLOR_CLAVE   = '#FF7043'   # naranja — elemento que se inserta
    COLOR_DESP    = '#FFF176'   # amarillo — elemento desplazado
    COLOR_SORTED  = '#A5D6A7'   # verde — prefijo ordenado

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(0, max(datos_orig) + 2)
    ax.set_title('Insertion Sort — animación', fontsize=13, fontweight='bold')
    ax.set_xlabel('Índice')
    ax.set_ylabel('Valor')
    ax.grid(axis='y', alpha=0.3)

    bars  = ax.bar(range(n), a, color=COLOR_BASE, edgecolor='white', linewidth=1.2)
    textos = [ax.text(i, a[i] + 0.2, str(a[i]), ha='center', va='bottom', fontsize=9)
              for i in range(n)]
    paso_txt = ax.text(0.02, 0.95, '', transform=ax.transAxes,
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    def actualizar(frame_idx):
        estado, pos_desp, pos_clave_orig, frontera = frames[frame_idx]
        for i, (bar, txt) in enumerate(zip(bars, textos)):
            bar.set_height(estado[i])
            txt.set_position((i, estado[i] + 0.2))
            txt.set_text(str(estado[i]))
            if i < frontera:
                bar.set_color(COLOR_SORTED)
            elif i == pos_desp:
                bar.set_color(COLOR_DESP)
            else:
                bar.set_color(COLOR_BASE)
        paso_txt.set_text(f'Paso {frame_idx}/{len(frames)-1}  |  frontera ordenada: {frontera}')
        return bars

    anim = animation.FuncAnimation(fig, actualizar, frames=len(frames),
                                   interval=interval, blit=False, repeat=False)
    plt.tight_layout()
    if EN_COLAB:
        display(HTML(anim.to_jshtml()))
    else:
        display(HTML(anim.to_jshtml()))
    plt.close()

random.seed(42)
datos = random.sample(range(1, 20), 8)
print(f"Datos a ordenar: {datos}")
animar_insertion_sort(datos, interval=600)

## Análisis de Complejidad — Insertion Sort

### Peor caso: lista ordenada inversamente

$$T_{\text{peor}}(n) = \sum_{i=1}^{n-1} i = \frac{n(n-1)}{2} = \Theta(n^2)$$

Cada elemento `a[i]` recorre todo el prefijo — **máximos desplazamientos**.

### Mejor caso: lista ya ordenada

$$T_{\text{mejor}}(n) = \sum_{i=1}^{n-1} 1 = n - 1 = \Theta(n)$$

Cada elemento `a[i]` no necesita desplazarse — **solo 1 comparación por paso**.

> 💡 **Insight clave:** Insertion Sort es el único de los tres algoritmos elementales que se beneficia del orden preexistente.  
> Para datos *casi ordenados* (pocas inversiones), es prácticamente lineal.

> ⚠️ **Importante:** En la práctica, Insertion Sort es el **algoritmo de elección para listas pequeñas** (n ≤ 20-30).  
> Librerías como CPython lo usan internamente dentro de Timsort para ese rango.

In [ ]:
# Benchmark: Insertion Sort en mejor, promedio y peor caso
import timeit
import random

def insertion_sort_simple(a):
    a = a[:]
    for i in range(1, len(a)):
        clave = a[i]
        j = i - 1
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = clave
    return a

ns = [100, 500, 1000, 2000]
print(f"{'n':>6} | {'Mejor (ordenado)':>18} | {'Promedio (random)':>18} | {'Peor (inverso)':>15}")
print("-" * 65)
for n in ns:
    mejor   = sorted(range(n))
    promedio = random.sample(range(n), n)
    peor    = list(range(n, 0, -1))

    t_mejor   = timeit.timeit(lambda: insertion_sort_simple(mejor),   number=50) / 50
    t_prom    = timeit.timeit(lambda: insertion_sort_simple(promedio), number=50) / 50
    t_peor    = timeit.timeit(lambda: insertion_sort_simple(peor),    number=50) / 50

    print(f"{n:>6} | {t_mejor*1000:>15.3f} ms | {t_prom*1000:>15.3f} ms | {t_peor*1000:>12.3f} ms")

print("\n→ Observa cómo el mejor caso escala linealmente (×2 cuando n dobla)")
print("  mientras el peor caso escala cuadráticamente (×4 cuando n dobla).")

# Sección 3: Bubble Sort — Repaso Formal (15 minutos)

## Lo que ya conocen (del assignment)

En el assignment anterior implementaron Bubble Sort como parte de la tarea.  
Aquí lo formalizamos y le agregamos la optimización que lo hace útil en práctica.

## La Idea

Bubble Sort recorre el arreglo comparando pares adyacentes y **burbujea** el máximo  
hacia el final en cada pasada:

```
Pasada 1: [5, 2, 8, 3] → compara (5,2)→swap, (5,8)→ok, (8,3)→swap → [2, 5, 3, 8]
Pasada 2: [2, 5, 3, 8] → compara (2,5)→ok, (5,3)→swap            → [2, 3, 5, 8]
Pasada 3: [2, 3, 5, 8] → compara (2,3)→ok                        → [2, 3, 5, 8] ✓
```

> 📌 **Invariante:** Tras la pasada `i`, los últimos `i` elementos están en su posición definitiva.

> 🎙️ **[PAUSA PROFESOR]** *"¿Cuál es la diferencia con Selection Sort? En ambos el sufijo queda fijo, pero ¿cómo?"*

In [ ]:
# Bubble Sort con early-exit (optimización con flag 'hubo_swap')
def bubble_sort(lista: list, verbose: bool = False) -> tuple:
    """
    Ordena 'lista' usando Bubble Sort con early-exit.

    La optimización con 'hubo_swap' permite terminar en O(n) si la lista
    ya está ordenada: si en una pasada completa no hubo ningún swap,
    el arreglo está ordenado y podemos salir.

    Parámetros:
        lista   (list): lista de elementos comparables
        verbose (bool): imprime el estado tras cada pasada

    Retorna:
        tuple: (lista_ordenada, n_comparaciones, n_swaps, n_pasadas)

    Complejidad:
        Temporal: O(n²) peor caso | O(n) mejor caso (con early-exit)
        Espacial: O(1) extra — in-place
    """
    a = lista[:]
    n = len(a)
    comparaciones = 0
    swaps         = 0
    pasada        = 0

    for i in range(n - 1):
        hubo_swap = False   # ← flag de early-exit
        pasada += 1

        for j in range(n - 1 - i):
            comparaciones += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
                hubo_swap = True

        if verbose:
            print(f"  Pasada {i+1}: {a}  (swaps esta pasada: {hubo_swap})")

        if not hubo_swap:   # ← EARLY EXIT: ya está ordenado
            if verbose:
                print(f"  → Early exit en pasada {i+1}: no hubo swaps")
            break

    return a, comparaciones, swaps, pasada


# ─── Demo ─────────────────────────────────────────────────────────────────
print("=== Bubble Sort — casi ordenado (early-exit útil) ===")
casi_ord = [1, 2, 4, 3, 5, 6, 7]
print(f"Entrada: {casi_ord}")
res, cmp, sw, pas = bubble_sort(casi_ord, verbose=True)
print(f"\nResultado: {res} | Pasadas: {pas}, Comparaciones: {cmp}, Swaps: {sw}")

print("\n=== Bubble Sort — peor caso (lista invertida) ===")
invertida = [7, 6, 5, 4, 3, 2, 1]
print(f"Entrada: {invertida}")
res2, cmp2, sw2, pas2 = bubble_sort(invertida, verbose=True)
print(f"\nResultado: {res2} | Pasadas: {pas2}, Comparaciones: {cmp2}, Swaps: {sw2}")

# Sección 4: Tabla Comparativa — Los 3 Algoritmos Elementales (5 minutos)

Esta tabla resume las propiedades que hemos estudiado en las últimas dos clases:

In [ ]:
# Tabla comparativa de los 3 algoritmos elementales
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

columnas = ['Algoritmo', 'Mejor caso', 'Caso promedio', 'Peor caso',
            'Swaps (peor)', 'Estable', 'Aprovecha orden']
filas = [
    ['Selection Sort', 'Θ(n²)',    'Θ(n²)',    'Θ(n²)',  'Θ(n) mínimo', '❌ No',  '❌ No'],
    ['Insertion Sort', 'Θ(n)',     'Θ(n²)',    'Θ(n²)',  'Θ(n²)',        '✅ Sí',  '✅ Sí'],
    ['Bubble Sort',    'Θ(n)*',    'Θ(n²)',    'Θ(n²)',  'Θ(n²)',        '✅ Sí',  '✅ Sí*'],
]

colores_filas = ['#E3F2FD', '#E8F5E9', '#FFF9C4']

tabla = ax.table(
    cellText=filas,
    colLabels=columnas,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(9.5)

# Estilo encabezado
for j in range(len(columnas)):
    tabla[(0, j)].set_facecolor('#37474F')
    tabla[(0, j)].set_text_props(color='white', fontweight='bold')

# Estilo filas
for i, color in enumerate(colores_filas, start=1):
    for j in range(len(columnas)):
        tabla[(i, j)].set_facecolor(color)

ax.set_title('Algoritmos de Ordenamiento Elemental — Comparación', 
             fontsize=12, fontweight='bold', pad=10)

notas = ax.text(0.5, -0.05, 
    "* Bubble Sort: mejor caso O(n) solo con la variante early-exit (flag hubo_swap)",
    ha='center', va='top', fontsize=8, color='#555', style='italic',
    transform=ax.transAxes)

plt.tight_layout()
plt.show()
print("\n💡 Para datos casi ordenados: Insertion Sort ≈ Bubble Sort (ambos O(n))")
print("   Para minimizar swaps (costosos): Selection Sort (máximo n-1 swaps)")
print("   Para uso general con n pequeño: Insertion Sort (práctica estándar)")

# Sección 5: Widget Interactivo — Comparador Visual

In [ ]:
# Widget: comparar Insertion Sort vs Bubble Sort en tiempo real
import ipywidgets as widgets
from IPython.display import display
import random, timeit

def insertion_sort_w(a):
    a = a[:]
    cmp = 0
    for i in range(1, len(a)):
        clave = a[i]; j = i - 1
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]; j -= 1; cmp += 1
        if j >= 0: cmp += 1
        a[j + 1] = clave
    return a, cmp

def bubble_sort_w(a):
    a = a[:]
    cmp = 0
    for i in range(len(a) - 1):
        hubo = False
        for j in range(len(a) - 1 - i):
            cmp += 1
            if a[j] > a[j+1]:
                a[j], a[j+1] = a[j+1], a[j]; hubo = True
        if not hubo: break
    return a, cmp

# ── controles ──
slider_n    = widgets.IntSlider(value=10, min=4, max=30, step=1,
                                description='n (tamaño):', style={'description_width': '150px'})
tipo_datos  = widgets.Dropdown(
    options=[('Aleatorio', 'random'), ('Ya ordenado', 'sorted'),
             ('Invertido', 'reversed'), ('Casi ordenado', 'nearly')],
    value='random', description='Tipo de datos:', style={'description_width': '150px'})
boton       = widgets.Button(description='▶ Comparar', button_style='primary')
salida      = widgets.Output()

def generar_datos(tipo, n):
    if tipo == 'random':   return random.sample(range(1, n*3), n)
    if tipo == 'sorted':   return list(range(1, n+1))
    if tipo == 'reversed': return list(range(n, 0, -1))
    if tipo == 'nearly':
        d = list(range(1, n+1))
        # intercambiar ~10% de pares
        for _ in range(max(1, n//10)):
            i, j = random.randint(0, n-2), random.randint(0, n-2)
            d[i], d[j] = d[j], d[i]
        return d

def al_comparar(b):
    with salida:
        salida.clear_output(wait=True)
        n    = slider_n.value
        tipo = tipo_datos.value
        datos = generar_datos(tipo, n)
        print(f"Datos ({tipo}, n={n}): {datos}")
        print()

        _, cmp_ins = insertion_sort_w(datos)
        _, cmp_bub = bubble_sort_w(datos)

        t_ins = timeit.timeit(lambda: insertion_sort_w(datos), number=500) / 500
        t_bub = timeit.timeit(lambda: bubble_sort_w(datos),   number=500) / 500

        print(f"{'Algoritmo':>16} | {'Comparaciones':>14} | {'Tiempo (ms)':>12}")
        print("-" * 48)
        print(f"{'Insertion Sort':>16} | {cmp_ins:>14} | {t_ins*1000:>10.4f}")
        print(f"{'Bubble Sort':>16} | {cmp_bub:>14} | {t_bub*1000:>10.4f}")

        if cmp_ins < cmp_bub:
            print(f"\n→ Insertion Sort hace {cmp_bub - cmp_ins} comparaciones MENOS")
        elif cmp_bub < cmp_ins:
            print(f"\n→ Bubble Sort hace {cmp_ins - cmp_bub} comparaciones MENOS")
        else:
            print("\n→ Mismo número de comparaciones")

boton.on_click(al_comparar)
display(widgets.VBox([
    widgets.HBox([slider_n, tipo_datos]),
    boton,
    salida
]))

# Sección 6: Ejercicios Prácticos

## 🧪 Ejercicio 1 ⭐: Insertion Sort Bidireccional (Cocktail Insertion)

**Descripción:** Implementa `insertion_sort_desc(lista)` que ordena la lista en orden  
**descendente** (de mayor a menor) usando Insertion Sort.

**Entrada:** Lista de enteros  
**Salida:** Nueva lista ordenada de mayor a menor

**Ejemplo:**
```
Entrada: [3, 1, 4, 1, 5, 9, 2]
Salida:  [9, 5, 4, 3, 2, 1, 1]
```

**Restricciones:** n ≤ 1000  
**Complejidad esperada:** O(n²) peor caso

In [ ]:
def insertion_sort_desc(lista: list) -> list:
    """
    Ordena 'lista' en orden descendente (mayor a menor) usando Insertion Sort.

    Parámetros:
        lista (list): lista de enteros
    Retorna:
        list: nueva lista ordenada descendentemente
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Verificador automático para insertion_sort_desc."""
    import time
    casos = [
        ([3, 1, 4, 1, 5, 9, 2],   [9, 5, 4, 3, 2, 1, 1],  "Caso con repetidos"),
        ([],                        [],                       "Lista vacía"),
        ([7],                       [7],                      "Un solo elemento"),
        ([1, 2, 3, 4, 5],          [5, 4, 3, 2, 1],          "Ya ordenado asc → invertir"),
        ([5, 4, 3, 2, 1],          [5, 4, 3, 2, 1],          "Ya ordenado desc → no cambiar"),
        ([2, 2, 2, 2],             [2, 2, 2, 2],             "Todos iguales"),
        (list(range(20, 0, -1)),   list(range(20, 0, -1)),   "n=20 peor caso"),
    ]
    aprobados = 0
    for args, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(args[:])  # copia para no mutar
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(insertion_sort_desc)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def insertion_sort_desc(lista: list) -> list:
#     """Insertion Sort en orden descendente — cambia solo el signo de comparación."""
#     a = lista[:]
#     for i in range(1, len(a)):
#         clave = a[i]
#         j = i - 1
#         # Cambio clave: a[j] < clave (en vez de >) para orden descendente
#         while j >= 0 and a[j] < clave:
#             a[j + 1] = a[j]
#             j -= 1
#         a[j + 1] = clave
#     return a
# 
# # Complejidad: O(n²) temporal, O(1) espacial extra
# # Nota: solo cambia el operador de comparación — el resto es idéntico

## 🧪 Ejercicio 2 ⭐⭐: Contar Inversiones

**Descripción:** Una **inversión** es un par de índices `(i, j)` con `i < j` y `a[i] > a[j]`.  
El número de inversiones mide cuán desordenada está una lista.

Implementa `contar_inversiones(lista)` que retorna el número total de inversiones.

**Ejemplo:**
```
Entrada: [3, 1, 2]
Inversiones: (3,1) y (3,2) → Salida: 2

Entrada: [1, 2, 3]
Salida: 0  (ya ordenado — sin inversiones)

Entrada: [3, 2, 1]
Salida: 3  (máximas inversiones para n=3)
```

**Restricciones:** n ≤ 500  
**Complejidad esperada:** O(n²) es suficiente  
**Pista:** ¿Cómo se relaciona el número de inversiones con el número de pasos de Insertion Sort?

In [ ]:
def contar_inversiones(lista: list) -> int:
    """
    Cuenta el número de inversiones en 'lista'.
    Una inversión es un par (i, j) con i < j y lista[i] > lista[j].

    Parámetros:
        lista (list): lista de enteros
    Retorna:
        int: número total de inversiones
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Verificador para contar_inversiones."""
    import time
    casos = [
        ([3, 1, 2],          2,  "Dos inversiones"),
        ([1, 2, 3],          0,  "Sin inversiones (ordenado)"),
        ([3, 2, 1],          3,  "Máximas inversiones n=3"),
        ([],                 0,  "Lista vacía"),
        ([5],                0,  "Un elemento"),
        ([2, 2, 2],          0,  "Todos iguales — no hay inversiones"),
        ([4, 3, 2, 1],       6,  "n=4 invertido: 6 inversiones"),
        ([1, 3, 2, 4, 5],    1,  "Una sola inversión"),
    ]
    aprobados = 0
    for args, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(args[:])
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc} | Esperado: {esperado} | Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(contar_inversiones)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def contar_inversiones(lista: list) -> int:
#     """
#     Cuenta inversiones con doble bucle O(n²).
#     
#     Conexión pedagógica: el número de inversiones es EXACTAMENTE el número
#     de desplazamientos que realizará Insertion Sort.
#     Es la medida natural de 'cuán desordenada' está la lista.
#     """
#     n = len(lista)
#     count = 0
#     for i in range(n):
#         for j in range(i + 1, n):
#             if lista[i] > lista[j]:
#                 count += 1
#     return count
# 
# # Verificación del vínculo: inversiones == desplazamientos de Insertion Sort
# # import random
# # datos = random.sample(range(100), 10)
# # _, _, desp = insertion_sort(datos)
# # inv = contar_inversiones(datos)
# # print(f"Desplazamientos IS: {desp} | Inversiones: {inv}")  # deben ser iguales

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Qué pasa si ordenas strings en vez de enteros? ¿Funciona sin cambios?
- ¿Puedes medir la relación exacta entre inversiones y desplazamientos de Insertion Sort?
- ¿Cuándo cambia el resultado del early-exit de Bubble Sort?

In [ ]:
# Espacio libre para experimentar
# Sugerencia: prueba insertion_sort(['banana', 'apple', 'cherry', 'date'])

In [ ]:
# Espacio libre para experimentar
# Sugerencia: genera listas con distintos porcentajes de desorden y mide comparaciones

# Sección 7: Autoevaluación

In [ ]:
import ipywidgets as widgets
from IPython.display import display

preguntas = [
    {
        "pregunta": "¿En qué caso Insertion Sort tiene complejidad O(n)?",
        "opciones": [
            "Cuando la lista está en orden inverso",
            "Cuando la lista ya está ordenada",
            "Cuando todos los elementos son iguales",
            "Nunca — siempre es O(n²)"
        ],
        "correcta": 1,
        "explicacion": "Con la lista ya ordenada, el while interno nunca se ejecuta: solo hay 1 comparación por iteración → n-1 comparaciones totales → O(n)."
    },
    {
        "pregunta": "¿Cuál es el propósito del flag 'hubo_swap' en Bubble Sort?",
        "opciones": [
            "Contar el número total de swaps realizados",
            "Terminar anticipadamente si en una pasada no hubo intercambios",
            "Evitar comparar elementos ya ordenados al inicio",
            "Detectar duplicados en la lista"
        ],
        "correcta": 1,
        "explicacion": "Si en una pasada completa no hubo ningún swap, el arreglo está ordenado. El flag permite detectar esto y salir en O(n) para listas ya ordenadas."
    },
    {
        "pregunta": "¿Cuál de los tres algoritmos elementales hace el MÍNIMO número de swaps?",
        "opciones": [
            "Insertion Sort",
            "Bubble Sort",
            "Selection Sort",
            "Los tres hacen el mismo número"
        ],
        "correcta": 2,
        "explicacion": "Selection Sort hace a lo sumo n-1 swaps (uno por pasada). Insertion Sort puede hacer O(n²) desplazamientos. Bubble Sort también puede hacer O(n²) swaps."
    },
    {
        "pregunta": "El número de desplazamientos de Insertion Sort es igual a:",
        "opciones": [
            "El número de elementos fuera de lugar",
            "n × log(n)",
            "El número de inversiones en la lista original",
            "Siempre n(n-1)/2"
        ],
        "correcta": 2,
        "explicacion": "Cada inversión (par i<j con a[i]>a[j]) corresponde exactamente a un desplazamiento de Insertion Sort. Es la conexión formal entre 'desorden' y costo del algoritmo."
    },
]

def crear_quiz(preguntas):
    for i, p in enumerate(preguntas):
        radio = widgets.RadioButtons(
            options=p["opciones"],
            description=f"P{i+1}:",
            style={'description_width': 'initial'},
            layout={'width': 'max-content'}
        )
        boton  = widgets.Button(description="Verificar", button_style="info")
        salida = widgets.Output()
        label  = widgets.HTML(f"<b>P{i+1}: {p['pregunta']}</b>")

        def verificar(b, r=radio, o=salida, c=p["correcta"], e=p["explicacion"], opts=p["opciones"]):
            with o:
                o.clear_output()
                if r.value == opts[c]:
                    print(f"✅ ¡Correcto! {e}")
                else:
                    print(f"❌ No exactamente. Respuesta: {opts[c]}")
                    print(f"   {e}")

        boton.on_click(verificar)
        display(widgets.VBox([label, radio, boton, salida]))
        print("─" * 70)

crear_quiz(preguntas)

# Sección 8: Lecturas y Recursos de Práctica

## Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Sedgewick & Wayne — *Algorithms* | 4ª ed. | Cap. 2.1 | Elementary Sorts: Insertion, Selection, Shell |
| Cormen et al. (CLRS) — *Intro to Algorithms* | 4ª ed. | Cap. 2.1 | Insertion Sort + loop invariants |
| Skiena — *The Algorithm Design Manual* | 3ª ed. | Cap. 4.1 | Applications of Sorting |

## Recursos gratuitos
- 🎬 [Insertion Sort — Visualgo](https://visualgo.net/en/sorting) — animación interactiva paso a paso
- 🎬 [Bubble Sort Dance — YouTube](https://www.youtube.com/watch?v=lyZQPjUT5B4) — visualización musical
- 📄 [Sedgewick — Ordenamiento elemental (slides)](https://algs4.cs.princeton.edu/21elementary/)

## Práctica en Codeforces

> 🔍 **Cómo filtrar:** [codeforces.com/problemset](https://codeforces.com/problemset) → Tags: `sortings` → Rating: 800–1200

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [Twins](https://codeforces.com/problemset/problem/160/A) | ⭐ 800 | Ordenar y tomar greedy — aplicación directa |
| 2 | [Nearly Sorted](https://codeforces.com/problemset/problem/1353/C) | ⭐ 900 | Detectar si una lista está casi ordenada |
| 3 | [Sort the Array](https://codeforces.com/problemset/problem/451/B) | ⭐⭐ 1000 | Ordenar subarray óptimo |
| 4 | [Inversion Count](https://codeforces.com/problemset/problem/368/C) | ⭐⭐⭐ 1500 | Contar inversiones eficientemente (Merge Sort) |

⚠️ Los problemas 1 y 2 son el **mínimo esperado**. El 3 es intermedio. El 4 es desafío opcional.